In [1]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import os
import subprocess
import requests
import time
import pandas as pd

print("Imports successful.")

Imports successful.


In [ ]:
# ============================================================

# STEP 4 PREREQUISITES — READ BEFORE RUNNING

# ============================================================

# This notebook requires two external tools that must be

# installed BEFORE running any cells:

#

# 1. OPEN BABEL — for molecular file format conversion

#    Install via conda:

#    conda install -c conda-forge openbabel

#

#    Verify installation by running in terminal:

#    obabel --version

#

# 2. AUTODOCK VINA v1.2.7 — for molecular docking

#    Download the executable for your operating system from:

#    https://github.com/ccsb-scripps/AutoDock-Vina/releases

#

#    Windows: download vina_1.2.7_win.exe

#             rename to vina.exe

#             place in a folder on your system PATH

#             or in the same folder as this notebook

#

#    Mac (Apple Silicon): download vina_1.2.7_mac_aarch64

#             chmod +x vina_1.2.7_mac_aarch64

#             move to ~/bin/vina (or any folder on PATH)

#

#    Linux:   download vina_1.2.7_linux_x86_64

#             chmod +x vina_1.2.7_linux_x86_64

#             sudo mv vina_1.2.7_linux_x86_64 /usr/local/bin/vina

#

#    Google Colab (Linux):

#             !wget https://github.com/ccsb-scripps/AutoDock-Vina/releases/download/v1.2.7/vina_1.2.7_linux_x86_64

#             !chmod +x vina_1.2.7_linux_x86_64

#             !mv vina_1.2.7_linux_x86_64 /usr/local/bin/vina

#

#    Verify installation by running in terminal:

#    vina --version

#

# 3. PYTHON ENVIRONMENT

#    This notebook was developed using Python 3.11

#    in a conda environment. Required packages:

#    pip install pubchempy pandas numpy requests

#    conda install -c conda-forge rdkit

#

# ============================================================

# Once both tools are installed and verified, proceed with

# running the cells below in order.

# ============================================================



print("Prerequisites cell — do not run until Open Babel and AutoDock Vina are installed.")

print("See comments above for installation instructions.")



In [2]:
# Load the shortlist from Step 3
df = pd.read_csv("orexin_shortlist_for_docking.csv")
print(f"Loaded {len(df)} compounds for docking.")
print(f"\nCompounds to dock:")
print(df[["name", "drug_class", "composite_score"]].to_string(index=False))

# Create folders to organize docking files
os.makedirs("receptors", exist_ok=True)
os.makedirs("ligands_3d", exist_ok=True)
os.makedirs("docking_results", exist_ok=True)

print(f"\nFolders created:")
print(f"  receptors/       — for OX1R and OX2R PDB files")
print(f"  ligands_3d/      — for 3D ligand structures")
print(f"  docking_results/ — for Vina output files")

Loaded 20 compounds for docking.

Compounds to dock:
                          name     drug_class  composite_score
  similar_to_56944144_67473934    DORA_analog         0.780719
  similar_to_56944144_56944522    DORA_analog         0.778408
  similar_to_56944144_56944044    DORA_analog         0.777564
  similar_to_56944144_67283149    DORA_analog         0.775976
  similar_to_56944144_56944620    DORA_analog         0.772382
  similar_to_56944144_67473922    DORA_analog         0.771801
  similar_to_56944144_56944429    DORA_analog         0.771063
                   Lemborexant           DORA         0.770513
  similar_to_56944144_56944045    DORA_analog         0.769788
 similar_to_24965990_137634720    DORA_analog         0.769677
  similar_to_56944144_56944245    DORA_analog         0.768800
  similar_to_56944144_56944043    DORA_analog         0.768312
 similar_to_24965990_137652411    DORA_analog         0.767556
  similar_to_56944144_67281908    DORA_analog         0.767246
  

In [3]:
# ============================================================
# Download orexin receptor structures from PDB
# ============================================================

def download_pdb(pdb_id, output_dir="receptors"):
    """Download a PDB structure from RCSB."""
    url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
    filepath = f"{output_dir}/{pdb_id}.pdb"
    
    if os.path.exists(filepath):
        print(f"  {pdb_id}.pdb already exists, skipping download.")
        return filepath
    
    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        with open(filepath, "w") as f:
            f.write(response.text)
        print(f"  Downloaded: {filepath}")
        return filepath
    except Exception as e:
        print(f"  Failed to download {pdb_id}: {e}")
        return None

print("Downloading orexin receptor structures...")
ox1r_path = download_pdb("4ZJC")   # OX1R with suvorexant
ox2r_path = download_pdb("4S0V")   # OX2R with suvorexant

print(f"\nReceptor files ready:")
print(f"  OX1R: {ox1r_path}")
print(f"  OX2R: {ox2r_path}")

  Downloaded: receptors/4ZJC.pdb
  Downloaded: receptors/4S0V.pdb

Receptor files ready:
  OX1R: receptors/4ZJC.pdb
  OX2R: receptors/4S0V.pdb


In [4]:
# ============================================================
# Prepare receptor files using Open Babel
# ============================================================

def prepare_receptor(pdb_id, input_dir="receptors"):
    """
    Clean a PDB file and convert to PDBQT format for AutoDock Vina.
    Removes waters and converts to the format Vina requires.
    """
    input_path  = f"{input_dir}/{pdb_id}.pdb"
    output_path = f"{input_dir}/{pdb_id}_prepared.pdbqt"

    if os.path.exists(output_path):
        print(f"  {pdb_id}_prepared.pdbqt already exists, skipping.")
        return output_path

    cmd = [
        "obabel",
        input_path,
        "-O", output_path,
        "-xr",          # remove all but largest fragment
        "-h",           # add hydrogens
        "--delete", "HOH",  # remove water molecules
    ]

    result = subprocess.run(cmd, capture_output=True, text=True)

    if os.path.exists(output_path):
        print(f"  Prepared: {output_path}")
        return output_path
    else:
        print(f"  Failed: {pdb_id}")
        print(f"  Error: {result.stderr}")
        return None

print("Preparing receptor files...")
ox1r_prepared = prepare_receptor("4ZJC")
ox2r_prepared = prepare_receptor("4S0V")

print(f"\nPrepared receptors:")
print(f"  OX1R: {ox1r_prepared}")
print(f"  OX2R: {ox2r_prepared}")

Preparing receptor files...
  Prepared: receptors/4ZJC_prepared.pdbqt
  Prepared: receptors/4S0V_prepared.pdbqt

Prepared receptors:
  OX1R: receptors/4ZJC_prepared.pdbqt
  OX2R: receptors/4S0V_prepared.pdbqt


In [5]:
# ============================================================
# Prepare ligand files using Open Babel
# ============================================================

def prepare_ligand(name, smiles, output_dir="ligands_3d"):
    """
    Convert a SMILES string to a 3D PDBQT file for docking.
    Steps: SMILES → 3D coordinates → energy minimize → PDBQT
    """
    # Create a safe filename by removing special characters
    safe_name = name.replace(" ", "_").replace("/", "-")
    output_path = f"{output_dir}/{safe_name}.pdbqt"

    if os.path.exists(output_path):
        print(f"  {safe_name} already exists, skipping.")
        return output_path

    cmd = [
        "obabel",
        f"-:{smiles}",      # input as SMILES string
        "-O", output_path,  # output file
        "--gen3d",          # generate 3D coordinates
        "--minimize",       # energy minimize
        "--ff", "MMFF94",   # force field
        "--steps", "500",   # minimization steps
        "-h",               # add hydrogens
    ]

    result = subprocess.run(cmd, capture_output=True, text=True)

    if os.path.exists(output_path):
        return output_path
    else:
        print(f"  Failed: {safe_name}")
        print(f"  Error: {result.stderr}")
        return None

print("Preparing ligand files...")
ligand_paths = {}

for _, row in df.iterrows():
    path = prepare_ligand(row["name"], row["clean_smiles"])
    if path:
        ligand_paths[row["name"]] = path
        print(f"  OK: {row['name']}")

print(f"\nSuccessfully prepared: {len(ligand_paths)} / {len(df)} ligands")

Preparing ligand files...
  OK: similar_to_56944144_67473934
  OK: similar_to_56944144_56944522
  OK: similar_to_56944144_56944044
  OK: similar_to_56944144_67283149
  OK: similar_to_56944144_56944620
  OK: similar_to_56944144_67473922
  OK: similar_to_56944144_56944429
  OK: Lemborexant
  OK: similar_to_56944144_56944045
  OK: similar_to_24965990_137634720
  OK: similar_to_56944144_56944245
  OK: similar_to_56944144_56944043
  OK: similar_to_24965990_137652411
  OK: similar_to_56944144_67281908
  OK: similar_to_56944144_67282848
  OK: similar_to_56944144_67473935
  OK: similar_to_154617563_137460857
  OK: similar_to_24965990_137638805
  OK: similar_to_56944144_56944143
  OK: similar_to_56944144_67282493

Successfully prepared: 20 / 20 ligands


In [6]:
# ============================================================
# Define binding boxes for OX1R and OX2R
# Centered on the suvorexant binding site from crystal structures
# ============================================================

BINDING_BOXES = {
    "OX1R": {
        "receptor": "receptors/4ZJC_prepared.pdbqt",
        "center_x": -1.4,
        "center_y":  3.5,
        "center_z": -2.8,
        "size_x": 20,
        "size_y": 20,
        "size_z": 20,
    },
    "OX2R": {
        "receptor": "receptors/4S0V_prepared.pdbqt",
        "center_x": -1.8,
        "center_y":  4.2,
        "center_z": -1.9,
        "size_x": 20,
        "size_y": 20,
        "size_z": 20,
    }
}

print("Binding boxes defined:")
for receptor, params in BINDING_BOXES.items():
    print(f"\n  {receptor}:")
    print(f"    Center: ({params['center_x']}, "
          f"{params['center_y']}, {params['center_z']})")
    print(f"    Box size: {params['size_x']} x "
          f"{params['size_y']} x {params['size_z']} Angstroms")

Binding boxes defined:

  OX1R:
    Center: (-1.4, 3.5, -2.8)
    Box size: 20 x 20 x 20 Angstroms

  OX2R:
    Center: (-1.8, 4.2, -1.9)
    Box size: 20 x 20 x 20 Angstroms


In [9]:
# ============================================================
# Run AutoDock Vina docking (fixed for Vina 1.2.7)
# ============================================================

def run_vina(ligand_name, ligand_path, receptor_name, box_params,
             exhaustiveness=8, num_modes=9):
    """
    Run AutoDock Vina for one ligand against one receptor.
    Returns the best binding score (most negative = tightest binding).
    """
    safe_name = ligand_name.replace(" ", "_").replace("/", "-")
    out_path  = f"docking_results/{safe_name}_{receptor_name}_out.pdbqt"

    cmd = [
        "vina",
        "--receptor", box_params["receptor"],
        "--ligand",   ligand_path,
        "--center_x", str(box_params["center_x"]),
        "--center_y", str(box_params["center_y"]),
        "--center_z", str(box_params["center_z"]),
        "--size_x",   str(box_params["size_x"]),
        "--size_y",   str(box_params["size_y"]),
        "--size_z",   str(box_params["size_z"]),
        "--out",      out_path,
        "--exhaustiveness", str(exhaustiveness),
        "--num_modes",      str(num_modes),
    ]

    result = subprocess.run(cmd, capture_output=True, text=True)

    # Save stdout to a log file manually
    log_path = f"docking_results/{safe_name}_{receptor_name}_log.txt"
    with open(log_path, "w") as f:
        f.write(result.stdout)
        f.write(result.stderr)

    # Parse best score from Vina output
    best_score = None
    for line in result.stdout.split("\n"):
        line = line.strip()
        if line.startswith("1 "):
            parts = line.split()
            try:
                best_score = float(parts[1])
            except (IndexError, ValueError):
                pass

    return best_score, out_path

# Run docking for all ligands against both receptors
print("Starting docking calculations...")
print("This will take 20-40 minutes. Progress will show below.\n")

results = []
total = len(ligand_paths) * len(BINDING_BOXES)
count = 0

for ligand_name, ligand_path in ligand_paths.items():
    for receptor_name, box_params in BINDING_BOXES.items():
        count += 1
        print(f"  [{count}/{total}] {ligand_name} vs {receptor_name}...")

        score, out_file = run_vina(
            ligand_name, ligand_path, receptor_name, box_params
        )

        results.append({
            "name":          ligand_name,
            "receptor":      receptor_name,
            "docking_score": score,
            "pose_file":     out_file,
        })

        if score is not None:
            print(f"    Score: {score} kcal/mol")
        else:
            print(f"    Score: could not parse")

print(f"\nDocking complete! {len(results)} calculations finished.")

Starting docking calculations...
This will take 20-40 minutes. Progress will show below.

  [1/40] similar_to_56944144_67473934 vs OX1R...
    Score: -5.64 kcal/mol
  [2/40] similar_to_56944144_67473934 vs OX2R...
    Score: 0.0 kcal/mol
  [3/40] similar_to_56944144_56944522 vs OX1R...
    Score: -5.521 kcal/mol
  [4/40] similar_to_56944144_56944522 vs OX2R...
    Score: 0.0 kcal/mol
  [5/40] similar_to_56944144_56944044 vs OX1R...
    Score: -5.756 kcal/mol
  [6/40] similar_to_56944144_56944044 vs OX2R...
    Score: 0.0 kcal/mol
  [7/40] similar_to_56944144_67283149 vs OX1R...
    Score: -5.919 kcal/mol
  [8/40] similar_to_56944144_67283149 vs OX2R...
    Score: 0.0 kcal/mol
  [9/40] similar_to_56944144_56944620 vs OX1R...
    Score: -5.835 kcal/mol
  [10/40] similar_to_56944144_56944620 vs OX2R...
    Score: 0.0 kcal/mol
  [11/40] similar_to_56944144_67473922 vs OX1R...
    Score: -6.004 kcal/mol
  [12/40] similar_to_56944144_67473922 vs OX2R...
    Score: 0.0 kcal/mol
  [13/40] simi

In [10]:
# Diagnose the OX2R docking failure
test_ligand_name = "Lemborexant"
test_ligand_path = ligand_paths[test_ligand_name]
box_params = BINDING_BOXES["OX2R"]

safe_name = test_ligand_name.replace(" ", "_")
out_path = f"docking_results/{safe_name}_OX2R_debug.pdbqt"

cmd = [
    "vina",
    "--receptor", box_params["receptor"],
    "--ligand",   test_ligand_path,
    "--center_x", str(box_params["center_x"]),
    "--center_y", str(box_params["center_y"]),
    "--center_z", str(box_params["center_z"]),
    "--size_x",   str(box_params["size_x"]),
    "--size_y",   str(box_params["size_y"]),
    "--size_z",   str(box_params["size_z"]),
    "--out",      out_path,
    "--exhaustiveness", "8",
    "--num_modes",      "9",
]

result = subprocess.run(cmd, capture_output=True, text=True)

print("STDOUT:")
print(result.stdout)
print("\nSTDERR:")
print(result.stderr)
print(f"\nReturn code: {result.returncode}")

STDOUT:
AutoDock Vina v1.2.7
#################################################################
# If you used AutoDock Vina in your work, please cite:          #
#                                                               #
# J. Eberhardt, D. Santos-Martins, A. F. Tillack, and S. Forli  #
# AutoDock Vina 1.2.0: New Docking Methods, Expanded Force      #
# Field, and Python Bindings, J. Chem. Inf. Model. (2021)       #
# DOI 10.1021/acs.jcim.1c00203                                  #
#                                                               #
# O. Trott, A. J. Olson,                                        #
# AutoDock Vina: improving the speed and accuracy of docking    #
# with a new scoring function, efficient optimization and       #
# multithreading, J. Comp. Chem. (2010)                         #
# DOI 10.1002/jcc.21334                                         #
#                                                               #
# Please see https://github.com/ccsb-scripps/Au

In [11]:
# Extract binding site coordinates from the co-crystallized ligand
# Suvorexant is labeled SXR in the PDB files

def get_ligand_center(pdb_file, ligand_code):
    """
    Parse a PDB file and calculate the center of mass
    of the co-crystallized ligand — this is our binding box center.
    """
    x_coords, y_coords, z_coords = [], [], []
    
    with open(pdb_file, "r") as f:
        for line in f:
            if (line.startswith("HETATM") and 
                ligand_code in line[17:20]):
                try:
                    x_coords.append(float(line[30:38]))
                    y_coords.append(float(line[38:46]))
                    z_coords.append(float(line[46:54]))
                except ValueError:
                    pass
    
    if not x_coords:
        print(f"  Ligand {ligand_code} not found in {pdb_file}")
        return None, None, None
    
    cx = sum(x_coords) / len(x_coords)
    cy = sum(y_coords) / len(y_coords)
    cz = sum(z_coords) / len(z_coords)
    
    return round(cx, 2), round(cy, 2), round(cz, 2)

print("Extracting binding site centers from crystal structures...")

# OX1R — 4ZJC
cx1, cy1, cz1 = get_ligand_center("receptors/4ZJC.pdb", "SXR")
print(f"\nOX1R (4ZJC) binding center:")
print(f"  X={cx1}, Y={cy1}, Z={cz1}")

# OX2R — 4S0V
cx2, cy2, cz2 = get_ligand_center("receptors/4S0V.pdb", "SXR")
print(f"\nOX2R (4S0V) binding center:")
print(f"  X={cx2}, Y={cy2}, Z={cz2}")

Extracting binding site centers from crystal structures...
  Ligand SXR not found in receptors/4ZJC.pdb

OX1R (4ZJC) binding center:
  X=None, Y=None, Z=None
  Ligand SXR not found in receptors/4S0V.pdb

OX2R (4S0V) binding center:
  X=None, Y=None, Z=None


In [12]:
# Find all HETATM ligand codes in both PDB files
def find_ligands(pdb_file):
    """Print all unique ligand codes found in a PDB file."""
    ligands = set()
    with open(pdb_file, "r") as f:
        for line in f:
            if line.startswith("HETATM"):
                code = line[17:20].strip()
                ligands.add(code)
    return ligands

print("Ligands found in 4ZJC (OX1R):")
print(find_ligands("receptors/4ZJC.pdb"))

print("\nLigands found in 4S0V (OX2R):")
print(find_ligands("receptors/4S0V.pdb"))

Ligands found in 4ZJC (OX1R):
{'OLA', '4OT'}

Ligands found in 4S0V (OX2R):
{'OLA', 'SUV', 'HOH'}


In [13]:
# Get correct binding centers using actual ligand codes
cx1, cy1, cz1 = get_ligand_center("receptors/4ZJC.pdb", "4OT")
print(f"OX1R (4ZJC) binding center:")
print(f"  X={cx1}, Y={cy1}, Z={cz1}")

cx2, cy2, cz2 = get_ligand_center("receptors/4S0V.pdb", "SUV")
print(f"\nOX2R (4S0V) binding center:")
print(f"  X={cx2}, Y={cy2}, Z={cz2}")

# Update the binding boxes with correct coordinates
BINDING_BOXES["OX1R"]["center_x"] = cx1
BINDING_BOXES["OX1R"]["center_y"] = cy1
BINDING_BOXES["OX1R"]["center_z"] = cz1

BINDING_BOXES["OX2R"]["center_x"] = cx2
BINDING_BOXES["OX2R"]["center_y"] = cy2
BINDING_BOXES["OX2R"]["center_z"] = cz2

print(f"\nBinding boxes updated with correct coordinates.")

OX1R (4ZJC) binding center:
  X=-7.49, Y=0.24, Z=-54.15

OX2R (4S0V) binding center:
  X=52.46, Y=7.97, Z=53.44

Binding boxes updated with correct coordinates.


In [14]:
# ============================================================
# Re-run docking with corrected binding box coordinates
# ============================================================

print("Re-running docking with corrected binding site coordinates...")
print(f"\nOX1R center: X={cx1}, Y={cy1}, Z={cz1}")
print(f"OX2R center: X={cx2}, Y={cy2}, Z={cz2}\n")

results = []
total = len(ligand_paths) * len(BINDING_BOXES)
count = 0

for ligand_name, ligand_path in ligand_paths.items():
    for receptor_name, box_params in BINDING_BOXES.items():
        count += 1
        print(f"  [{count}/{total}] {ligand_name} vs {receptor_name}...")

        score, out_file = run_vina(
            ligand_name, ligand_path, receptor_name, box_params
        )

        results.append({
            "name":          ligand_name,
            "receptor":      receptor_name,
            "docking_score": score,
            "pose_file":     out_file,
        })

        if score is not None:
            print(f"    Score: {score} kcal/mol")
        else:
            print(f"    Score: could not parse")

print(f"\nDocking complete! {len(results)} calculations finished.")

Re-running docking with corrected binding site coordinates...

OX1R center: X=-7.49, Y=0.24, Z=-54.15
OX2R center: X=52.46, Y=7.97, Z=53.44

  [1/40] similar_to_56944144_67473934 vs OX1R...
    Score: -8.766 kcal/mol
  [2/40] similar_to_56944144_67473934 vs OX2R...
    Score: -5.712 kcal/mol
  [3/40] similar_to_56944144_56944522 vs OX1R...
    Score: -8.555 kcal/mol
  [4/40] similar_to_56944144_56944522 vs OX2R...
    Score: -4.414 kcal/mol
  [5/40] similar_to_56944144_56944044 vs OX1R...
    Score: -9.057 kcal/mol
  [6/40] similar_to_56944144_56944044 vs OX2R...
    Score: -3.635 kcal/mol
  [7/40] similar_to_56944144_67283149 vs OX1R...
    Score: -7.756 kcal/mol
  [8/40] similar_to_56944144_67283149 vs OX2R...
    Score: -4.688 kcal/mol
  [9/40] similar_to_56944144_56944620 vs OX1R...
    Score: -7.608 kcal/mol
  [10/40] similar_to_56944144_56944620 vs OX2R...
    Score: -4.775 kcal/mol
  [11/40] similar_to_56944144_67473922 vs OX1R...
    Score: -8.872 kcal/mol
  [12/40] similar_to_

In [15]:
# ============================================================
# Compile and save docking results
# ============================================================

docking_df = pd.DataFrame(results)

# Pivot so OX1R and OX2R scores are side by side
docking_wide = docking_df.pivot_table(
    index="name",
    columns="receptor",
    values="docking_score"
).reset_index()

docking_wide.columns.name = None
docking_wide = docking_wide.rename(columns={
    "OX1R": "score_OX1R",
    "OX2R": "score_OX2R"
})

# Calculate average and selectivity
docking_wide["avg_docking_score"] = (
    docking_wide[["score_OX1R", "score_OX2R"]].mean(axis=1)
)
docking_wide["selectivity_OX1_vs_OX2"] = (
    docking_wide["score_OX1R"] - docking_wide["score_OX2R"]
)

# Merge with shortlist properties
shortlist_props = pd.read_csv("orexin_shortlist_for_docking.csv")
final_df = docking_wide.merge(
    shortlist_props[["name", "drug_class", "molecular_weight",
                     "xlogp", "tpsa", "composite_score",
                     "BBB_Martins", "hERG"]],
    on="name", how="left"
)

# Sort by best average docking score
final_df = final_df.sort_values("avg_docking_score")

print("Final docking results:")
print(final_df[["name", "drug_class", "score_OX1R",
                 "score_OX2R", "avg_docking_score",
                 "selectivity_OX1_vs_OX2"]].to_string(index=False))

# Save
final_df.to_csv("final_docking_results.csv", index=False)
print(f"\nSaved: final_docking_results.csv")

Final docking results:
                          name     drug_class  score_OX1R  score_OX2R  avg_docking_score  selectivity_OX1_vs_OX2
  similar_to_56944144_56944043    DORA_analog      -8.796     -7.7800           -8.28800                 -1.0160
                   Lemborexant           DORA      -9.113     -6.4460           -7.77950                 -2.6670
  similar_to_56944144_56944429    DORA_analog      -8.741     -6.7810           -7.76100                 -1.9600
  similar_to_56944144_67473934    DORA_analog      -8.766     -5.7120           -7.23900                 -3.0540
  similar_to_56944144_56944245    DORA_analog      -8.921     -5.4920           -7.20650                 -3.4290
  similar_to_56944144_67473935    DORA_analog      -8.349     -5.5730           -6.96100                 -2.7760
  similar_to_56944144_67282493    DORA_analog      -7.203     -6.4570           -6.83000                 -0.7460
similar_to_154617563_137460857 agonist_analog      -8.056     -5.4850    